# 02 — Content-Based Music Recommendation

This notebook demonstrates the baseline content-based recommender.

**Architecture:**
```
songs_1m.parquet -> StandardScaler -> content_features.npy -> cosine similarity -> Top-N
```

**Artifacts used:**
- `data/processed/songs_1m.parquet` — source dataset
- `data/processed/content_features.npy` — normalised feature matrix
- `data/processed/song_index.parquet` — row_index -> song metadata
- `data/processed/content_scaler.joblib` — fitted StandardScaler

In [1]:
import sys
import os

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Processed Data

In [2]:
DATA_DIR = os.path.join("..", "data", "processed")

# Feature matrix
X = np.load(os.path.join(DATA_DIR, "content_features.npy"))
print(f"Feature matrix: {X.shape}  dtype={X.dtype}")

# Song index
song_index = pd.read_parquet(os.path.join(DATA_DIR, "song_index.parquet"))
print(f"Song index:     {song_index.shape}")

# Scaler
scaler = joblib.load(os.path.join(DATA_DIR, "content_scaler.joblib"))
print(f"Scaler:         {scaler.__class__.__name__} with {len(scaler.mean_)} features")

# Original dataset (for feature comparison)
df = pd.read_parquet(os.path.join(DATA_DIR, "songs_1m.parquet"))
print(f"Source dataset: {df.shape}")

Feature matrix: (1000000, 13)  dtype=float64
Song index:     (1000000, 4)
Scaler:         StandardScaler with 13 features
Source dataset: (1000000, 27)


## 2. Inspect Feature Matrix

In [3]:
print(f"Shape:  {X.shape}")
print(f"Dtype:  {X.dtype}")
print(f"Size:   {X.nbytes / (1024**2):.1f} MB")
print()
print("Sample vector (row 0):")
print(X[0])

Shape:  (1000000, 13)
Dtype:  float64
Size:   99.2 MB

Sample vector (row 0):
[-0.45922656 -0.0451352   0.1937891  -0.56638411 -0.10657152 -0.46915264
 -0.44786555  0.17099949  1.24277321  0.45386344  0.19967232 -1.22383444
 -2.03603501]


In [4]:
CONTENT_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration", "key", "mode", "time_signature",
]

feature_stats = pd.DataFrame({
    "feature": CONTENT_FEATURES,
    "scaler_mean": scaler.mean_,
    "scaler_std": scaler.scale_,
    "scaled_mean": X.mean(axis=0),
    "scaled_std": X.std(axis=0),
})
feature_stats

,feature,scaler_mean,scaler_std,scaled_mean,scaled_std
0,danceability,0.602740,0.177994,1.455192e-16,1.0
1,energy,0.603149,0.247023,3.037712e-16,1.0
2,loudness,-8.995287,6.286666,2.164597e-16,1.0
3,speechiness,0.107331,0.135828,-4.729372e-17,1.0
4,acousticness,0.342147,0.320417,-5.275069e-17,1.0
5,instrumentalness,0.144214,0.307393,-4.001777e-17,1.0
6,liveness,0.197325,0.168187,5.456968e-18,1.0
7,valence,0.490627,0.259492,-1.318767e-16,1.0
8,tempo,120.700370,30.013223,-2.164597e-16,1.0
9,duration,207.227281,96.444690,-5.070433e-17,1.0


## 3. Load the Recommender

In [5]:
# Change working directory temporarily so the recommender finds its files
original_cwd = os.getcwd()
os.chdir(os.path.join(".."))

from src.models.content_model import ContentRecommender

rec = ContentRecommender()
print(f"Loaded: {len(rec.feature_matrix):,d} songs, {rec.feature_matrix.shape[1]} features")

Loaded: 1,000,000 songs, 13 features


## 4. Song 1 — Believer by Imagine Dragons

In [6]:
info_1 = rec.get_song_info("0pqnGHJpmpxLKifKRmU6WP")
print(f"Track:  {info_1['track_name']}")
print(f"Artist: {info_1['artist_name']}")
print(f"ID:     {info_1['track_id']}")

results_1 = rec.recommend_by_track_id("0pqnGHJpmpxLKifKRmU6WP", top_n=10)

pd.DataFrame(results_1).style.format({"similarity_score": "{:.4f}"})

Track:  Believer
Artist: Imagine Dragons
ID:     0pqnGHJpmpxLKifKRmU6WP


,track_id,track_name,artist_name,similarity_score
0,3cgAt9gbx3cWz9tpvJXneZ,Prom Night,fromis_9,0.9933
1,04ONyebJMyt11q4IBBv4pt,Believer,Imagine Dragons,0.9927
2,2YgKJbcFbdhpTIfypGD9nF,Believer,Imagine Dragons,0.9927
3,63vlNTXQxLuRV122wxzzcV,Believer,Imagine Dragons,0.9927
4,10tEoFeBxVm4z1CjsocAUH,Sexy Lady,Magixx,0.9906
5,7Mbwkrtni29Th2zhRk1zZv,ŻÓŁTA CZAPKA,BSK,0.9875
6,5cWgu5sSCDJOahIkLceJKX,Los Fuletes Estan Ready,Flamen Beretta,0.9866
7,1xVpaTfaxNXhz8gxIZKnbm,Ja Bih Se S Tobom Topila,Mia Borisavljevic,0.9862
8,6WrI0LAC5M1Rw2MnX2ZvEg,Don't Start Now,Dua Lipa,0.9862
9,2RevV77RPztZ3I4snuCzGU,MOONWALKER,SEVENTEEN,0.9859


## 5. Song 2 — Die With A Smile by Bruno Mars

In [7]:
info_2 = rec.get_song_info("2plbrEY59IikOBgBGLjaoe")
print(f"Track:  {info_2['track_name']}")
print(f"Artist: {info_2['artist_name']}")
print(f"ID:     {info_2['track_id']}")

results_2 = rec.recommend_by_track_id("2plbrEY59IikOBgBGLjaoe", top_n=10)

pd.DataFrame(results_2).style.format({"similarity_score": "{:.4f}"})

Track:  Die With A Smile
Artist: Bruno Mars
ID:     2plbrEY59IikOBgBGLjaoe


,track_id,track_name,artist_name,similarity_score
0,78mE33YpoWqMQTwgUQRhyo,Die With A Smile,Bruno Mars,0.9997
1,7nEHTOQJqKCx7Dvgng3l8t,Die With A Smile,Bruno Mars,0.9988
2,7so0lgd0zP2Sbgs2d7a1SZ,Die With A Smile,Bruno Mars,0.9985
3,1Ywk7HecwsfJWy2pxaaQZM,Die With A Smile,Bruno Mars,0.9897
4,2wyr7VKEVMwXxhx6fOSbOe,TE EXTRAÑO Y LO SIENTO...,DannyLux,0.9794
5,4VBdklDplNfHUFTuvQlNE6,Senga Soola Kaara,Ghibran,0.9642
6,2GOVAi24NKeojRx2p7DnDa,La trama principal,Dani Fernández,0.9625
7,6DFhHn7m2Q48WCwgmJgDgS,Teri Deewani - MTV Unwind,Kailash Kher,0.9621
8,0XkPkRaJcQf4b8uvl5i4ff,Paradise,Maher Zain,0.9611
9,5GZ6liz6FVhzAx8VQKCSbe,Paradise,Maher Zain,0.9611


## 6. Song 3 — BIRDS OF A FEATHER by Billie Eilish

In [8]:
info_3 = rec.get_song_info("6dOtVTDdiauQNBQEDOtlAB")
print(f"Track:  {info_3['track_name']}")
print(f"Artist: {info_3['artist_name']}")
print(f"ID:     {info_3['track_id']}")

results_3 = rec.recommend_by_track_id("6dOtVTDdiauQNBQEDOtlAB", top_n=10)

pd.DataFrame(results_3).style.format({"similarity_score": "{:.4f}"})

Track:  BIRDS OF A FEATHER
Artist: Billie Eilish
ID:     6dOtVTDdiauQNBQEDOtlAB


,track_id,track_name,artist_name,similarity_score
0,6uMOteLgUpMjkDQcCsoyo5,BIRDS OF A FEATHER,Billie Eilish,0.9896
1,1dbNg2ougivyUv6f9hQQzU,Kwijt,Suzan & Freek,0.9843
2,22nV4UdZiQ4XZq9UDbfTu8,Kwijt,Suzan & Freek,0.9839
3,7K8aUdJE60bBl3Sxkz8euy,Get High,King Lil G,0.9721
4,4R2xhVHmdz4vD2WSJFKKeT,Santhosey,SHAN PUTHA,0.9663
5,2ShIjRVUa6B4INiSvp4BvT,Forever Young,Danny Darko,0.9638
6,40M76o0ihzPKPxF97L4Zcz,Flashing Lights,Kansy,0.9629
7,3o4qlkbRP4DSXl4NBrWULJ,Quiero,Shakira,0.9628
8,5jLlvGprnTXbWbSlyWIHGd,Subtle Chime,Ellard,0.9613
9,2yDdPMIlUuSTLoprkozAaV,Fools Direction - 2014 Remaster,Michael Learns To Rock,0.9608


## 7. Feature Comparison — Sanity Check

Compare the raw audio features of the query song against its top recommendations
to verify that similar songs are actually close in feature space.

In [9]:
# Use Believer as the test case
query_id = "0pqnGHJpmpxLKifKRmU6WP"
query_row = rec._id_to_row[query_id]
query_features = df.iloc[query_row][CONTENT_FEATURES]

print("Believer — Raw Features:")
print(query_features.to_string())
print()

# Compare with top 3 recommendations
results = rec.recommend_by_track_id(query_id, top_n=3)

comparison_rows = [{"song": "[INPUT] Believer", **query_features.to_dict()}]

for i, r in enumerate(results, 1):
    row_idx = rec._id_to_row[r["track_id"]]
    feats = df.iloc[row_idx][CONTENT_FEATURES]
    comparison_rows.append({
        "song": f"[{i}] {r['track_name']}",
        **feats.to_dict()
    })

comparison = pd.DataFrame(comparison_rows).set_index("song")
comparison

Believer — Raw Features:
danceability         0.776
energy                0.78
loudness            -4.374
speechiness          0.128
acousticness        0.0622
instrumentalness       0.0
liveness             0.081
valence              0.666
tempo                  125
duration               204
key                     10
mode                     0
time_signature           4



,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,key,mode,time_signature
song,,,,,,,,,,,,,
[INPUT] Believer,0.776,0.780,-4.374,0.128,0.0622,0.0,0.0810,0.666,125,204,10,0,4
[1] Prom Night,0.787,0.837,-3.308,0.157,0.0325,0.0,0.0605,0.667,125,206,10,0,4
[2] Believer,0.766,0.775,-4.267,0.141,0.0504,0.0,0.1280,0.695,125,204,10,0,4
[3] Believer,0.766,0.775,-4.267,0.141,0.0504,0.0,0.1280,0.695,125,204,10,0,4


In [10]:
# Restore working directory
os.chdir(original_cwd)
print("Done.")

Done.


## Summary

The content-based recommender successfully generates Top-N similar songs
using cosine similarity over normalised audio features.

**Limitations:**
- Brute-force similarity is not optimised for production-scale request volumes.
- Duplicate track_name+artist_name pairs (different track_ids) appear in results.
- No ground-truth relevance data exists to compute accuracy metrics.

**Next step:** FAISS / approximate nearest-neighbour search for efficient similarity retrieval.